In [386]:
import os, pandas as pd

In [387]:
metadatos = ['id_ubicacion', 'especie', 'tratamiento', 'salinidad', 'nombre_archivo']
datos_trackeo = ['t', 'x', 'y', 'vx', 'vy', 'v', 'ax', 'ay', 'a']
datos_resumen = ['v_promedio', 'desplazamiento', 'curv_promedio', 'curv_mediana', 'despl_relativo']

def agregar_id(df, metadatos):
    df.loc[:, 'id'] = df.groupby(metadatos, observed = True).ngroup()
    columnas = ['id'] + [col for col in df.columns if col != 'id']
    return df.loc[:, columnas]

def mapeo_columnas(df, mapeo):
    for columna, diccionario in mapeo.items():
        if columna in df.columns:
            df.loc[:, columna] = df[columna].map(diccionario)
        else:
            print("Columna no encontrada: " + columna)
    return df

def filtrar_df_resumen(df, metadatos, medidas_resumen):
    agregar_id(df, metadatos)
    df = df.groupby('id', observed = True).first()
    return df.dropna(subset = medidas_resumen, how='any').reset_index(drop = True)

def definir_dtypes(df):
    tipos = {
        'id': 'category',
        'id_ubicacion': 'category',
        'especie': 'category',
        'tratamiento': 'category',
        'salinidad': 'category',
        'nombre_archivo': 'string',
        'estadio_larval': 'category',
        'obs': 'category',
        'video_analizado': 'category',
        'larva_unica': 'category',
    }
    columnas_faltantes = [col for col in tipos if col not in df.columns]
    if columnas_faltantes:
        print(f"Estas columnas no están en el DataFrame: {columnas_faltantes}")
    
    tipos_filtrados = {col: dtype for col, dtype in tipos.items() if col in df.columns}
    return df.astype(tipos_filtrados)

def añadir_larva_unica(df, metadatos):
    combinaciones = {
        (2, 'CC', 'E', '20', 'Video_004'), (2, 'CC', 'E', '25', 'Video_001'),
        (2, 'CC', 'E', '25', 'Video_003'), (2, 'CC', 'E', '25', 'Video_006'),
        (2, 'CC', 'E', '30', 'Video_003'), (2, 'CC', 'E', '35', 'Video_002'),
        (2, 'CC', 'E', '35', 'Video_003'), (2, 'CC', 'E', '35', 'Video_004'),
        (2, 'CC', 'E', '35', 'Video_006'), (2, 'CC', 'R', '05', 'Video_001'),
        (2, 'CC', 'R', '05', 'Video_002'), (2, 'CC', 'R', '05', 'Video_004'),
        (2, 'CC', 'R', '05', 'Video_005'), (2, 'CC', 'R', '05', 'Video_006'),
        (2, 'CC', 'R', '10', 'Video_001'), (2, 'CC', 'R', '10', 'Video_002'),
        (2, 'CC', 'R', '10', 'Video_003'), (2, 'CC', 'R', '10', 'Video_005'),
        (2, 'CC', 'R', '25', 'Video_001'), (2, 'CC', 'R', '25', 'Video_002'),
        (2, 'CC', 'R', '30', 'Video_001'), (2, 'CC', 'R', '30', 'Video_004'),
        (2, 'CC', 'R', '30', 'Video_006'), (2, 'CC', 'R', '35', 'Video_002'),
        (2, 'CC', 'R', '35', 'Video_004'), (2, 'LA', 'E', '20', 'Video_005'),
        (2, 'LA', 'E', '25', 'Video_001'), (2, 'LA', 'E', '30', 'Video_001'),
        (2, 'LA', 'E', '30', 'Video_003'), (2, 'LA', 'E', '30', 'Video_004'),
        (2, 'LA', 'E', '30', 'Video_006'), (2, 'LA', 'E', '35', 'Video_002'),
        (2, 'LA', 'E', '35', 'Video_003'), (2, 'LA', 'E', '35', 'Video_005'),
        (2, 'LA', 'E', '35', 'Video_006'), (2, 'LA', 'R', '15', 'Video_003'),
        (2, 'LA', 'R', '20', 'Video_005'), (2, 'LA', 'R', '25', 'Video_002'),
        (2, 'LA', 'R', '30', 'Video_003'), (2, 'LA', 'R', '30', 'Video_004'),
        (2, 'LA', 'R', '35', 'Video_001'), (2, 'LA', 'R', '35', 'Video_005'),
        (2, 'LA', 'R', '35', 'Video_006'), (3, 'LA', 'E', '25', 'Video_005'),
        (3, 'LA', 'E', '30', 'Video_001'), (3, 'LA', 'E', '35', 'Video_002'),
        (3, 'LA', 'E', '35', 'Video_003'), (3, 'LA', 'E', '35', 'Video_004'),
        (3, 'LA', 'E', '35', 'Video_005'), (3, 'LA', 'E', '35', 'Video_006'),
        (3, 'LA', 'R', '15', 'Video_004'), (3, 'LA', 'R', '20', 'Video_001'),
        (3, 'LA', 'R', '20', 'Video_002'), (3, 'LA', 'R', '20', 'Video_005'),
        (3, 'LA', 'R', '25', 'Video_003'), (3, 'LA', 'R', '25', 'Video_004'),
        (3, 'LA', 'R', '25', 'Video_005'), (3, 'LA', 'R', '30', 'Video_002'),
        (3, 'LA', 'R', '30', 'Video_004'), (3, 'LA', 'R', '30', 'Video_005'),
        (3, 'LA', 'R', '35', 'Video_002')}

    df.loc[:, 'larva_unica'] = 1
    tuplas_filas = df[metadatos].apply(tuple, axis=1)
    df.loc[tuplas_filas.isin(combinaciones), 'larva_unica'] = 0
    return df

Lectura de los datos de trackeo desde los .txt no vacíos (321) de cada video.

In [388]:
ruta = os.path.join("..", "..", "data", "Análisis Bioensayos", "01 - Datos de trackeo")

df_lista = []

for root, dirs, files in os.walk(ruta):
    for file in files:
        if file.endswith('.txt'):
            file_path = os.path.join(root, file)
            temp_df = pd.read_csv(file_path, delimiter=',')

            ubicacion, especie, tratamiento, salinidad, video = file_path.split(os.sep)[-6:-1]
            
            temp_df['id_ubicacion'] = ubicacion
            temp_df['especie'] = especie
            temp_df['tratamiento'] = tratamiento
            temp_df['salinidad'] = salinidad
            temp_df['nombre_archivo'] = video
            
            df_lista.append(temp_df)
    
trackeo_df = pd.concat(df_lista, ignore_index=True)
trackeo_df.drop('Var1', axis=1, inplace=True) #eliminamos la columna Var1 que se genera por los tres videos sin datos (2CCR05003, 2CCR05005, 2CCE25005)

#diccionarios para renombrar valores al formato estándar
ids_ubicaciones = {'01 - Candelaria 2023-08': 1,
                '02 - APR Quintay 2023-10A': 2,
                '03 - APR Quintay 2023-10B': 3}

especies = {'CC - Locos': 'CC',
        'LA - Erizo': 'LA'}

tratamientos = {'E - Exposición': 'E',
            'R - Recuperación': 'R'}
    
mapeos_trackeo = {'id_ubicacion': ids_ubicaciones,
                  'especie': especies,
                  'tratamiento': tratamientos}

trackeo_df = mapeo_columnas(trackeo_df, mapeos_trackeo)
trackeo_df

,t,x,y,vx,vy,v,ax,ay,a,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo
0,0.000000,1.976667,-0.624416,NaN,NaN,NaN,NaN,NaN,NaN,1,CC,E,35-N,Video_001
1,0.073174,1.981245,-0.629966,-0.030119,-0.122312,0.125966,NaN,NaN,NaN,1,CC,E,35-N,Video_001
2,0.123811,1.972938,-0.639559,-0.059602,-0.092332,0.109898,-0.117845,0.754723,0.763868,1,CC,E,35-N,Video_001
3,0.181472,1.974790,-0.639965,0.000456,-0.021718,0.021723,1.482924,0.656206,1.621626,1,CC,E,35-N,Video_001
4,0.224938,1.972984,-0.641756,0.041020,-0.068821,0.080119,0.133586,-0.786954,0.798211,1,CC,E,35-N,Video_001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95619,15.362240,2.660554,-3.178143,-0.062044,-0.000735,0.062048,-0.453923,0.212576,0.501233,3,LA,R,35,Video_006
95620,15.418690,2.656156,-3.176854,-0.088877,0.020600,0.091234,-0.036891,0.124915,0.130249,3,LA,R,35,Video_006
95621,15.463080,2.651592,-3.176066,-0.067448,0.006764,0.067787,0.335484,-0.099799,0.350014,3,LA,R,35,Video_006
95622,15.510260,2.649979,-3.176235,-0.041035,0.006275,0.041512,NaN,NaN,NaN,3,LA,R,35,Video_006


Lectura del archivo de Excel 'ListaVideos' para generar el dataframe con medidas de cada video; de este se anexarán algunas columnas a *df_trackeo*, mientras que las demás se dejarán en el dataframe aparte *df_resumen*.

In [389]:
ubicaciones = ['01 - Candelaria', '02 - APR Quintay A', '03 - APR Quintay B']
hojas = pd.read_excel(os.path.join("..", "..", "data", "Análisis Bioensayos", "ListaVideos.xlsx"), sheet_name = ubicaciones, header = 4)

ubicaciones_df = []

for nombre_hoja, df in hojas.items():
    df = df[['Nombre Común ', 'Estadío Larval', 'Tratamiento', 'Salinidad [PSU]', 
            'Nombre archivo', 'Observación', 'Video Analizado', 'Velocidad Promedio [mm/s]', 
            'Desplazamiento [mm]', 'Curvatura Promedio [rad/mm]', 'Curvatura Mediana [rad/mm]', 
            'Desplazamiento Relativo']].copy()
    df['id_ubicacion'] = nombre_hoja
    df.rename(columns = {'Nombre Común ': 'especie', 
                        'Estadío Larval': 'estadio_larval',
                        'Tratamiento': 'tratamiento',
                        'Salinidad [PSU]': 'salinidad',
                        'Nombre archivo': 'nombre_archivo',
                        'Observación': 'obs',
                        'Video Analizado': 'video_analizado',
                        'Velocidad Promedio [mm/s]': 'v_promedio',
                        'Desplazamiento [mm]': 'desplazamiento',
                        'Curvatura Promedio [rad/mm]': 'curv_promedio',
                        'Curvatura Mediana [rad/mm]': 'curv_mediana',
                        'Desplazamiento Relativo': 'despl_relativo'}, inplace=True)
    ubicaciones_df.append(df)
    
resumen_df = pd.concat(ubicaciones_df, ignore_index=True)

#diccionarios para renombrar valores igual que en df_trackeo
ids_ubicaciones_2 = {'01 - Candelaria': 1,
                    '02 - APR Quintay A': 2,
                    '03 - APR Quintay B': 3}

especies_2 = {'Loco': 'CC',
            'Erizo rojo': 'LA'}

tratamientos_2 = {'Exposición': 'E',
                'Recuperación': 'R'}

salinidades = {5: '05',
               10: '10',
               15: '15',
               20: '20',
               25: '25',
               30: '30',
               35: '35',
               '35-N': '35-N',
               '35-R': '35-R',
               40: '40',
               45: '45',
               50: '50',
               55: '55',
               60: '60'}

estadios_larvales = {'larva recien eclosionada': 'Larva recién eclosionada',
                'larva equinopluteus': 'Larva equinopluteus',
                'larva prisma': 'Larva prisma',
                'larva equinoplueus': 'Larva equinopluteus'} #correccion de nombres como vienen en el Excel

video_analizado = {'Si': 1, 
                   'No':0}

obs = {0:0, 1:1, 2:2,
       'no hay molusco': 3,
       'el video es 4x': 3,
       'molusco sale rápido': 3} #tercera categoría, para los tres archivos que no tienen datos

mapeos_resumen = {
    'id_ubicacion': ids_ubicaciones_2,
    'especie': especies_2,
    'tratamiento': tratamientos_2,
    'salinidad': salinidades,
    'estadio_larval': estadios_larvales,
    'video_analizado': video_analizado,
    'obs': obs
}

#para mover id_ubicacion a segunda posición
columnas_resumen = resumen_df.columns.tolist()
id_ubi = columnas_resumen.pop(-1)
columnas_resumen.insert(1, id_ubi)
resumen_df = resumen_df[columnas_resumen]

resumen_df = mapeo_columnas(resumen_df, mapeos_resumen)

In [390]:
columnas_merge = list(set(resumen_df.columns).intersection(trackeo_df.columns))

formato = trackeo_df[columnas_merge].copy() #para recuperar las mayúsculas luego de transformar para el merge

for col in columnas_merge:
    trackeo_df[col] = trackeo_df[col].astype(str).str.strip().str.lower()
    resumen_df[col] = resumen_df[col].astype(str).str.strip().str.lower()

df_trackeo = pd.merge(trackeo_df, resumen_df.drop(columns = datos_resumen), #quitamos las métricas de resumen; estas quedan en un df aparte
                       how = 'left', on = columnas_merge)

df_resumen = pd.merge(trackeo_df.drop(columns = datos_trackeo), resumen_df,
                      how = 'left', on = columnas_merge)

for col in columnas_merge:
    df_trackeo[col] = formato[col] #recuperamos las mayúsculas y espacios originales
    df_resumen[col] = formato[col]

In [391]:
#toques finales de cada df

df_trackeo = df_trackeo.copy() #copias de las vistas para evitar warnings
df_resumen = df_resumen.copy()

df_trackeo = (
    df_trackeo
    .pipe(añadir_larva_unica, metadatos)
    .pipe(agregar_id, metadatos)
    .pipe(definir_dtypes))

df_resumen = (
    df_resumen
    .pipe(filtrar_df_resumen, metadatos, datos_resumen)
    .pipe(añadir_larva_unica, metadatos)
    .pipe(agregar_id, metadatos)
    .pipe(definir_dtypes))

In [392]:
df_trackeo.dtypes

id                       category
t                         float64
x                         float64
y                         float64
vx                        float64
vy                        float64
v                         float64
ax                        float64
ay                        float64
a                         float64
id_ubicacion             category
especie                  category
tratamiento              category
salinidad                category
nombre_archivo     string[python]
estadio_larval           category
obs                      category
video_analizado          category
larva_unica              category
dtype: object

In [393]:
df_resumen.dtypes

id                       category
id_ubicacion             category
especie                  category
tratamiento              category
salinidad                category
nombre_archivo     string[python]
estadio_larval           category
obs                      category
video_analizado          category
v_promedio                float64
desplazamiento            float64
curv_promedio             float64
curv_mediana              float64
despl_relativo            float64
larva_unica              category
dtype: object

In [ ]:
#df_resumen.to_csv(os.path.join("..", "..", "data", "Datos resumen.csv"), index = False)
#df_trackeo.to_csv(os.path.join("..", "..", "data", "Datos trackeo.csv"), index = False)